# Project 1: Model Experimentation, Tuning & Explainability

## 1. Overview
In this notebook, we evaluate candidate classification models (Logistic Regression, Random Forest, XGBoost) using Stratified 5-Fold Cross-Validation, perform hyperparameter tuning, evaluate on the test set, and inspect SHAP feature attributions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("../src"))
from data_loader import load_raw_data, clean_data, get_feature_target_split
from preprocessor import get_preprocessor_pipeline
from train import build_candidate_pipelines, evaluate_cross_validation

df_raw = load_raw_data()
df_clean = clean_data(df_raw)
X, y = get_feature_target_split(df_clean)
print(f"Feature matrix: {X.shape}, Target vector: {y.shape}")

## 2. Cross-Validation Model Benchmark
Compare Logistic Regression, Random Forest, and XGBoost using Stratified 5-Fold CV across ROC-AUC, F1, Recall, Precision, and Accuracy.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

candidates = build_candidate_pipelines()
cv_results = evaluate_cross_validation(candidates, X_train, y_train)
cv_results

## 3. Threshold Optimization
Default 0.50 decision threshold is often sub-optimal for churn where missing a churner (False Negative) has high business cost. Let's analyze Precision-Recall trade-offs.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Fit top pipeline on train
top_model = candidates["XGBoost"]
top_model.fit(X_train, y_train)
y_probs = top_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recall, precision, color="purple", lw=2, label=f"PR Curve (AUC = {auc(recall, precision):.3f})")
axes[0].set_title("Precision-Recall Curve")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].legend()

axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.show()